# 刷题阶段总结(星期四, 2020年4月16日 10:17:32)

- [x] 0-1背包问题
- [x] 编辑距离
- [x] 数独-回溯法求解
- [x] KMP算法


## 0-1背包问题

(极致压缩dp求解法)

In [4]:
# 物品信息
v,w = [1,2,3],[6,10,12]
# 物品总量,背包限制
n,C = len(v),5 
# 量化容积
dp = [0]*(C+1)
# 初始化
for c in range(v[0],C+1):
    dp[c] = w[0]
# 状态求解
for i in range(1,n):
    for c in range(C,v[i]-1,-1):
        dp[c] = max(dp[c],dp[c-v[i]]+w[i])
# 最优价值
dp[-1] 

22

背包问题求解集合等价值划分


`v = [1,5,11,5]`
=> `[1,5,5]`+`[11]`

此时
- 质量和价值使用一组数据表示
- 背包的限制是 `SUM//2`
- 要求是最大价值和`背包质量`,即总和一般相同

该方法还可以用于求解集合最小划分差

In [6]:
v = [1,5,11,5]
SUM = sum(v)
n,C =len(v),SUM // 2
# require(C%2==0)
dp = [0]*(C+1)
for c in range(v[0],C+1):
    dp[c] = v[0]

for i in range(1,n):
    for c in range(C,v[i]-1,-1):
        dp[c] = max(dp[c],dp[c-v[i]]+v[i])

dp[-1]

11

## 编辑距离

动态规划从尾向头算

- 递归
- 动规

In [9]:
def ld(str1,str2):
    if str1 == '':
        return len(str2)
    if str2 == '':
        return len(str1)
    elif str1 == str2:
        return 0
    
    if str1[-1] == str2[-1]:
        d = 0
    else:
        d = 1 
    
    return min(
        ld(str1[:-1],str2)+1,
        ld(str1,str2[:-1])+1,
        ld(str1[:-1],str2[:-1])+d
    )

ld("abc","abcd")

1

In [12]:
def ldp(str1,str2):
    l1,l2 = len(str1),len(str2)
    dp = [[0]*(l2+1) for _ in range(l1+1)]
    
    for i in range(1,l1+1):
        for j in range(1,l2+1):
            if str1[i-1] == str2[j-1]:
                d = 0
            else:
                d =1
            dp[i][j] = min(
                dp[i-1][j]+1,
                dp[i][j-1]+1,
                dp[i-1][j-1]+d
            )
    return dp[-1][-1]

ldp("abc","abcd")

1

## 数独-回溯法

In [29]:
# def check_sudoku(sudoku):
sudoku = [
  ["5","3",".",".","7",".",".",".","."],
  ["6",".",".","1","9","5",".",".","."],
  [".","9","8",".",".",".",".","6","."],
  ["8",".",".",".","6",".",".",".","3"],
  ["4",".",".","8",".","3",".",".","1"],
  ["7",".",".",".","2",".",".",".","6"],
  [".","6",".",".",".",".","2","8","."],
  [".",".",".","4","1","9",".",".","5"],
  [".",".",".",".","8",".",".","7","9"]
]

def map2cell(row,col):
    if 0 <= row <= 2:  
        if 0 <= col <= 2: return 0
        elif 3 <= col <= 5: return 1
        elif 6 <= col <= 8: return 2
    elif 3 <= row <= 5:
        if 0 <= col <= 2: return 3
        elif 3 <= col <= 5: return 4
        elif 6 <= col <= 8: return 5
    elif 6 <= row <= 8:
        if 0 <= col <= 2: return 6
        elif 3 <= col <= 5: return 7
        elif 6 <= col <= 8: return 8

cols  = [[0]*9 for _ in range(9)] # 
rows  = [[0]*9 for _ in range(9)] # 
cells = [[0]*9 for _ in range(9)] #

# 信息采取
for i in range(9):
    for j in range(9):
        if sudoku[i][j]!='.':
            cur = int(sudoku[i][j])-1
            rows[i][cur] += 1
            cols[j][cur] += 1
            cells[map2cell(i,j)][cur] += 1

# 热量计算
row_sum = []
col_sum = []
cellsum = []
for i in range(9):
    row_sum.append(rows[i])
    col_sum.append(cols[i])
    cellsum.append(cells[i])

heats = [] 
for i in range(9):
    for j in range(9):
        heat = row_sum[i]+col_sum[j]+cellsum[map2cell(i,j)]
        if sudoku[i][j]=='.':
            heats.append((heat,i,j))
            
heats = sorted(heats,reverse=True)
# 按序解决待解决点
def solver(idx):
    if idx < 0:
        return False
    if idx == len(heats):
        return True
    heat,row,col = heats[idx]
    possible_val = []
    for v in range(9):
        if rows[row][v] == 0 and cols[col][v]==0 and cells[map2cell(row,col)][v] == 0:
            possible_val.append(v)
    for v in possible_val:
        str_v = str(v+1)
        sudoku[row][col] = str_v
        rows[row][v] +=1 
        cols[col][v] +=1
        cells[map2cell(row,col)][v] +=1 
        if solver(idx+1):return True
        rows[row][v] -=1 
        cols[col][v] -=1
        cells[map2cell(row,col)][v] -=1 
        sudoku[row][col] = '.'
    
    return False

solver(0)

True

## KMP算法

- next的生成(-1开始,头元素设置为-1,方便找到起点)
- next的应用,都是0开始

In [42]:
def getNext(str_pattern):
    i, j, n = 0, -1, len(str_pattern)
    next = [0]*n
    next[0] = -1
    
    while i < n-1:
        if j==-1 or str_pattern[i] == str_pattern[j]:
            i += 1
            j += 1
            next[i] = j
        else: 
            j = next[j]
    return next

def kmp(str_match,str_pattern):
    i, j, n, next = 0, 0, len(str_match), getNext(str_pattern)
    while i < n and j < len(next):
        if j == -1 or str_match[i] == str_pattern[j]:
            i += 1
            j += 1
        else: j = next[j]
 
    if j == len(next):
        return i - len(next)
    else:
        return -1
    
# next = getNext("abcabcd")
print(next)
kmp("abdabc","abc")

[-1, 0, 0, 0, 1, 2, 3]


3

## 快速排序

In [9]:
def quickSort(arr,l,r):
    if l < r :
        mid = quickCore(arr,l,r)
        quickSort(arr,l,mid-1)
        quickSort(arr,mid+1,r)
        
def quickCore(arr,l,r):
    pviot = arr[l]
    while l < r:
        while l < r and arr[r] >= pviot:
            r -= 1
        arr[l] = arr[r]
        
        while l < r and arr[l] <= arr[r]:
            l+=1
        arr[r] = arr[l]
    
    arr[l] = pviot
    return l
    
l = [7,1,2,3,4,5,60,7,8,9,9]
quickSort(l,0,len(l)-1)
l

[1, 2, 3, 4, 5, 7, 7, 8, 9, 9, 60]